In [ ]:
# ============================================================
# 1. Imports + CUDA debugging
# ============================================================

import os

# يجب أن يكون قبل استخدام CUDA
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import pandas as pd

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    BertTokenizer,
    BertForSequenceClassification
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import kagglehub


# ============================================================
# 2. Download Dataset
# ============================================================

path = kagglehub.dataset_download(
    "abhi8923shriv/sentiment-analysis-dataset"
)

print("Path to dataset files:", path)


# ============================================================
# 3. Load Dataset
# ============================================================

df = pd.read_csv(
    os.path.join(path, "train.csv"),
    encoding="latin-1"
)

print("Original shape:", df.shape)

print(df.head())


# ============================================================
# 4. Keep only required columns
# ============================================================

df = df[["text", "sentiment"]]

# Remove missing values
df = df.dropna()

# Remove duplicate rows
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

print("\nAfter cleaning:")
print("Dataset size:", len(df))

print("\nClass distribution:")
print(df["sentiment"].value_counts())


# ============================================================
# 5. Create Label Mapping
# ============================================================

labels = sorted(df["sentiment"].unique())

label2id = {
    label: idx
    for idx, label in enumerate(labels)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

print("\nlabel2id:")
print(label2id)

print("\nid2label:")
print(id2label)


# ============================================================
# 6. Validate Labels
# ============================================================

assert len(label2id) == 3

assert set(label2id.values()) == {0, 1, 2}

print("\nLabel validation passed.")


# ============================================================
# 7. Train / Test Split
# ============================================================

train_text, test_text, train_label, test_label = train_test_split(
    df["text"],
    df["sentiment"],
    test_size=0.25,
    shuffle=True,
    random_state=42,
    stratify=df["sentiment"]
)

print("\nTrain size:", len(train_text))
print("Test size:", len(test_text))


# ============================================================
# 8. Dataset Class
# ============================================================

class SentimentDataset(Dataset):

    def __init__(
        self,
        text,
        labels,
        tokenizer,
        max_len,
        label2id
    ):

        self.text = list(text)
        self.labels = list(labels)

        self.tokenizer = tokenizer
        self.max_len = max_len
        self.label2id = label2id

    def __len__(self):

        return len(self.text)

    def __getitem__(self, index):

        # -----------------------------
        # Text
        # -----------------------------

        current_text = str(self.text[index])

        # -----------------------------
        # Label
        # -----------------------------

        current_label = self.labels[index]

        label_id = self.label2id[current_label]

        # -----------------------------
        # Tokenization
        # -----------------------------

        encoding = self.tokenizer(
            current_text,

            add_special_tokens=True,

            max_length=self.max_len,

            padding="max_length",

            truncation=True,

            return_tensors="pt"
        )

        # -----------------------------
        # Return
        # -----------------------------

        return {
            "input_ids": encoding["input_ids"].squeeze(0),

            "attention_mask": encoding["attention_mask"].squeeze(0),

            "labels": torch.tensor(
                label_id,
                dtype=torch.long
            )
        }


# ============================================================
# 9. Build Model
# ============================================================

class BuildModel:

    def __init__(self):

        print("Preparing......")

        # ----------------------------------------------------
        # Configuration
        # ----------------------------------------------------

        self.epochs = 2

        self.max_len = 128

        self.batch_size = 32

        self.learning_rate = 1e-5

        self.model_name = "bert-base-uncased"

        # ----------------------------------------------------
        # Device
        # ----------------------------------------------------

        self.device = torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

        print("Device:", self.device)

        # ----------------------------------------------------
        # Tokenizer
        # ----------------------------------------------------

        print("\nLoading tokenizer...")

        self.tokenizer = BertTokenizer.from_pretrained(
            self.model_name
        )

        # ----------------------------------------------------
        # Model
        # ----------------------------------------------------

        print("\nLoading model...")

        self.model = BertForSequenceClassification.from_pretrained(
            self.model_name,

            num_labels=len(label2id),

            id2label=id2label,

            label2id=label2id
        )

        # ----------------------------------------------------
        # Model / Tokenizer validation
        # ----------------------------------------------------

        embedding_size = (
            self.model
            .get_input_embeddings()
            .num_embeddings
        )

        print("\nTokenizer length:", len(self.tokenizer))

        print(
            "Tokenizer vocab_size:",
            self.tokenizer.vocab_size
        )

        print(
            "Model embedding size:",
            embedding_size
        )

        print(
            "Number of labels:",
            self.model.config.num_labels
        )

        # ----------------------------------------------------
        # Dataset
        # ----------------------------------------------------

        self.train_dataset = SentimentDataset(
            text=train_text,

            labels=train_label,

            tokenizer=self.tokenizer,

            max_len=self.max_len,

            label2id=label2id
        )

        self.test_dataset = SentimentDataset(
            text=test_text,

            labels=test_label,

            tokenizer=self.tokenizer,

            max_len=self.max_len,

            label2id=label2id
        )

        # ----------------------------------------------------
        # DataLoader
        # ----------------------------------------------------

        self.train_dataloader = DataLoader(
            self.train_dataset,

            batch_size=self.batch_size,

            shuffle=True
        )

        self.test_dataloader = DataLoader(
            self.test_dataset,

            batch_size=self.batch_size,

            shuffle=False
        )

        # ----------------------------------------------------
        # Validate a single sample BEFORE CUDA
        # ----------------------------------------------------

        print("\nChecking first sample...")

        sample = self.train_dataset[0]

        input_ids = sample["input_ids"]

        attention_mask = sample["attention_mask"]

        label = sample["labels"]

        print(
            "input_ids shape:",
            input_ids.shape
        )

        print(
            "input_ids dtype:",
            input_ids.dtype
        )

        print(
            "input_ids min:",
            input_ids.min().item()
        )

        print(
            "input_ids max:",
            input_ids.max().item()
        )

        print(
            "attention_mask shape:",
            attention_mask.shape
        )

        print(
            "label:",
            label.item()
        )

        # ----------------------------------------------------
        # Validate input IDs
        # ----------------------------------------------------

        assert input_ids.dtype == torch.long

        assert input_ids.min().item() >= 0

        assert (
            input_ids.max().item()
            < embedding_size
        ), (
            f"Invalid token ID: "
            f"{input_ids.max().item()} "
            f">= embedding size "
            f"{embedding_size}"
        )

        # ----------------------------------------------------
        # Validate label
        # ----------------------------------------------------

        assert label.dtype == torch.long

        assert label.item() >= 0

        assert (
            label.item()
            < self.model.config.num_labels
        )

        print("\nSingle sample validation passed.")

        # ----------------------------------------------------
        # Optimizer
        # ----------------------------------------------------

        self.optimizer = AdamW(
            self.model.parameters(),

            lr=self.learning_rate
        )

        # ----------------------------------------------------
        # Move model to device
        # ----------------------------------------------------

        print("\nMoving model to device...")

        self.model.to(self.device)

        print("Model successfully moved to:", self.device)


    # ========================================================
    # Training
    # ========================================================

    def train(self):

        print("\nStarting training...\n")

        self.model.train()

        for epoch in range(self.epochs):

            print(
                f"\n========== Epoch "
                f"{epoch + 1}/{self.epochs} =========="
            )

            total_loss = 0.0

            progress_bar = tqdm(
                self.train_dataloader,
                desc=f"Epoch {epoch + 1}"
            )

            for batch in progress_bar:

                # ------------------------------------------------
                # Get batch
                # ------------------------------------------------

                input_ids = batch["input_ids"]

                attention_mask = batch["attention_mask"]

                labels = batch["labels"]

                # ------------------------------------------------
                # Validate CPU batch BEFORE moving to GPU
                # ------------------------------------------------

                embedding_size = (
                    self.model
                    .get_input_embeddings()
                    .num_embeddings
                )

                # input IDs

                assert input_ids.dtype == torch.long

                assert input_ids.min().item() >= 0

                assert (
                    input_ids.max().item()
                    < embedding_size
                ), (
                    f"Invalid input ID: "
                    f"{input_ids.max().item()} "
                    f">= embedding size "
                    f"{embedding_size}"
                )

                # labels

                assert labels.dtype == torch.long

                assert labels.min().item() >= 0

                assert (
                    labels.max().item()
                    < self.model.config.num_labels
                ), (
                    f"Invalid label: "
                    f"{labels.max().item()} "
                    f">= num_labels "
                    f"{self.model.config.num_labels}"
                )

                # ------------------------------------------------
                # Move batch to device
                # ------------------------------------------------

                input_ids = input_ids.to(
                    self.device
                )

                attention_mask = attention_mask.to(
                    self.device
                )

                labels = labels.to(
                    self.device
                )

                # ------------------------------------------------
                # Forward pass
                # ------------------------------------------------

                outputs = self.model(
                    input_ids=input_ids,

                    attention_mask=attention_mask,

                    labels=labels
                )

                loss = outputs.loss

                # ------------------------------------------------
                # Backward
                # ------------------------------------------------

                self.optimizer.zero_grad()

                loss.backward()

                self.optimizer.step()

                # ------------------------------------------------
                # Statistics
                # ------------------------------------------------

                total_loss += loss.item()

                progress_bar.set_postfix(
                    loss=f"{loss.item():.4f}"
                )

            # ----------------------------------------------------
            # Epoch loss
            # ----------------------------------------------------

            average_loss = (
                total_loss
                / len(self.train_dataloader)
            )

            print(
                f"Epoch {epoch + 1} "
                f"Average Loss: "
                f"{average_loss:.4f}"
            )


    # ========================================================
    # Evaluation
    # ========================================================

    def evaluate(self):

        print("\nEvaluating model...\n")

        self.model.eval()

        predictions = []

        true_labels = []

        with torch.no_grad():

            for batch in tqdm(
                self.test_dataloader,
                desc="Evaluation"
            ):

                input_ids = batch["input_ids"].to(
                    self.device
                )

                attention_mask = batch[
                    "attention_mask"
                ].to(self.device)

                labels = batch["labels"].to(
                    self.device
                )

                outputs = self.model(
                    input_ids=input_ids,

                    attention_mask=attention_mask
                )

                logits = outputs.logits

                batch_predictions = torch.argmax(
                    logits,
                    dim=1
                )

                predictions.extend(
                    batch_predictions.cpu().tolist()
                )

                true_labels.extend(
                    labels.cpu().tolist()
                )

        # ----------------------------------------------------
        # Accuracy
        # ----------------------------------------------------

        accuracy = accuracy_score(
            true_labels,
            predictions
        )

        print(
            f"\nAccuracy: {accuracy:.4f}"
        )

        # ----------------------------------------------------
        # Classification Report
        # ----------------------------------------------------

        print("\nClassification Report:\n")

        print(
            classification_report(
                true_labels,

                predictions,

                labels=list(
                    range(len(label2id))
                ),

                target_names=[
                    id2label[i]
                    for i in range(len(label2id))
                ],

                zero_division=0
            )
        )


    # ========================================================
    # Prediction
    # ========================================================

    def predict(self, text):

        self.model.eval()

        encoding = self.tokenizer(
            text,

            add_special_tokens=True,

            max_length=self.max_len,

            padding="max_length",

            truncation=True,

            return_tensors="pt"
        )

        input_ids = encoding[
            "input_ids"
        ].to(self.device)

        attention_mask = encoding[
            "attention_mask"
        ].to(self.device)

        with torch.no_grad():

            outputs = self.model(
                input_ids=input_ids,

                attention_mask=attention_mask
            )

        prediction_id = torch.argmax(
            outputs.logits,
            dim=1
        ).item()

        prediction_label = id2label[
            prediction_id
        ]

        return prediction_label


# ============================================================
# 10. Create Model
# ============================================================

model = BuildModel()


# ============================================================
# 11. Train
# ============================================================

model.train()


# ============================================================
# 12. Evaluate
# ============================================================

model.evaluate()


# ============================================================
# 13. Test Prediction
# ============================================================

text = "I really love this product"

prediction = model.predict(text)

print("\nText:", text)

print("Prediction:", prediction)
